<div class="thumbnail">
    <img src="https://williamtheisen.com/nd-cse-10124-lectures/Lecture_Images/Lecture06/slide-001.png" class="img-responsive"/>
</div>



In [ ]:
import math

def distance(a, b):
    return None

print(distance((2, 2), (1, 1)))

<div class="thumbnail">
    <img src="https://williamtheisen.com/nd-cse-10124-lectures/Lecture_Images/Lecture06/slide-002.png" class="img-responsive"/>
</div>

<div class="thumbnail">
    <img src="https://williamtheisen.com/nd-cse-10124-lectures/Lecture_Images/Lecture06/slide-003.png" class="img-responsive"/>
</div>


In [ ]:
import os

try:
    import google.colab
    REPO_URL = "https://github.com/wtheisen/nd-cse-10124-lectures.git"

    REPO_NAME = "/content/nd-cse-10124-lectures"
    L_PATH = "nd-cse-10124-lectures/Datasets"

    %cd /content/
    !rm -r {REPO_NAME}

    # Clone repo
    if not os.path.exists(REPO_NAME):
        !git clone {REPO_URL}

        # cd into the data folder
        %cd {L_PATH}
        !pwd

except ImportError:
    print("Unable to download repo, either:")
    print("\tA.) You're not on colab")
    print("\tB.) It has already been cloned")


#import utilities as uts

/content
Cloning into 'nd-cse-10124-lectures'...
remote: Enumerating objects: 287, done.
remote: Counting objects: 100% (20/20), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 287 (delta 8), reused 16 (delta 6), pack-reused 267 (from 1)
Receiving objects: 100% (287/287), 26.65 MiB | 41.61 MiB/s, done.
Resolving deltas: 100% (184/184), done.
/content/nd-cse-10124-lectures/Datasets
/content/nd-cse-10124-lectures/Datasets
Loading GloVe vectors (300d)...
Done.


In [1]:
import torch
from transformers import AutoTokenizer, AutoModel

def cos_sim(a, b):
    return torch.nn.functional.cosine_similarity(a, b, dim=0).item()

def token_embedding(word):
    # If the word splits into multiple wordpieces, you'll get multiple vectors.
    ids = tokenizer(word, add_special_tokens=False)["input_ids"]
    toks = tokenizer.convert_ids_to_tokens(ids)
    vecs = E[ids]  # (num_pieces, hidden_dim)
    return toks, ids, vecs

def project(word):
    _, _, v = token_embedding(word)
    return torch.dot(v[0], gender_axis).item()

model_name = "roberta-large"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.eval()

# embedding matrix: (vocab_size, hidden_dim)
E = model.get_input_embeddings().weight.detach().cpu()

t_dirty, ids_dirty, v_dirty = token_embedding("dirty")
print(v_dirty)
t_clean, ids_clean, v_clean = token_embedding("clean")
t_apple, ids_apple, v_apple = token_embedding("apple")

# Prompt: 'dirty'
# Vocab: {'dirty': 117}
# Embed: {117: tensor([[-0.0685, -0.2527,  0.0342,  ..., -0.0179, -0.0024,  0.0214]])}

# single-token words in roberta-large => one vector each
print(t_dirty, t_clean, t_apple)
print("dirty vs clean:", cos_sim(v_dirty[0], v_clean[0]))
print("dirty vs apple:", cos_sim(v_dirty[0], v_apple[0]))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


tensor([[-0.0685, -0.2527,  0.0342,  ..., -0.0179, -0.0024,  0.0214]])
['dirty'] ['clean'] ['apple']
dirty vs clean: 0.6328537464141846
dirty vs apple: 0.40697360038757324


In [2]:
# Install once (Colab-friendly)
!pip install -q gensim

import gensim.downloader as api
import numpy as np
import matplotlib.pyplot as plt

# ----------------
# Load GloVe
# ----------------
print("Loading GloVe vectors (300d)...")
model = api.load("glove-wiki-gigaword-300")
print("Done.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 43.1 MB/s eta 0:00:00
Loading GloVe vectors (300d)...
[==================================================] 100.0% 376.1/376.1MB downloaded
Done.


In [4]:
# ================================
# Historical NLP Bias Demonstration
# Using GloVe (2014)
# ================================
def cosine(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def project(word):
    v = model[word]
    return np.dot(v, gender_axis)

gender_axis = model["woman"] - model["man"]
gender_axis = gender_axis / np.linalg.norm(gender_axis)

# ----------------
# Test words
# ----------------
words = [
    "doctor",
    "nurse",
    "engineer",
    "scientist",
    "programmer",
    "teacher",
    "receptionist",
    "homemaker",
    "hfkakdmgkw"
]

values = [(w, project(w)) for w in words]
values.sort(key=lambda x: x[1])

print("\nProjection onto gender axis (negative = masculine, positive = feminine):\n")
for w, v in values:
    print(f"{w:15s} {v: .3f}")

words = [w for w, _ in values]
vals = np.array([v for _, v in values])

plt.figure(figsize=(9, 4))
plt.axvline(0.0, linewidth=1)
plt.scatter(vals, np.zeros_like(vals), s=80)

for w, v in zip(words, vals):
    plt.text(v, 0.02, w, ha="center", rotation=30)

plt.yticks([])
plt.xlabel("Gender direction (woman − man)")
plt.title("Gender Bias in Historical Word Embeddings (GloVe 2014)")
plt.tight_layout()
plt.show()

KeyError: "Key 'hfkakdmgkw' not present"

## Export to HTML

Uncomment the final line of the cell below and run it to export this notebook to HTML

In [ ]:
import os, json

def export_notebook():
  L_PATH = "nd-cse-10124-lectures/Notebooks"
  L = "Lecture_06_Embeddings_01"

  try:
      from google.colab import _message, files

      # where you WANT it to live (repo folder)
      repo_ipynb_path = f"/content/{L_PATH}/{L}.ipynb"

      # grab current notebook contents from the UI
      nb = _message.blocking_request("get_ipynb", timeout_sec=1)["ipynb"]

      # write it into the repo folder as a real file
      os.makedirs(os.path.dirname(repo_ipynb_path), exist_ok=True)
      with open(repo_ipynb_path, "w", encoding="utf-8") as f:
          json.dump(nb, f)

      # convert + download pdf
      !jupyter nbconvert --to html "{repo_ipynb_path}"
      files.download(repo_ipynb_path.replace(".ipynb", ".html"))
  except:
      import subprocess

      nb_fp = os.getcwd() + f'{L}.ipynb'
      print(os.getcwd())

      subprocess.run(["jupyter", "nbconvert", "--to", "html", nb_fp], check=True)

#export_notebook()